# agent0 fast-path demo

Walks through the fast-path pipeline stage by stage on a handful of example records.
No Modal deploy required — uses each stage as a plain Python function. The final cell
calls `process_record.local(...)` so the whole `@app.function`-decorated body runs
in-process without any remote dispatch or auth.

## Setup

Installs deps and puts the parent of `agent0/` on `sys.path` so the package is importable.
Works locally and in Colab as long as the repo (or at least `src/agent0/`) is somewhere
above this notebook on disk.

In [ ]:
import subprocess, sys, pathlib

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "modal>=0.64", "biopython>=1.83"]
)

nb_dir = pathlib.Path.cwd()
for p in [nb_dir, *nb_dir.parents]:
    if (p / "agent0").is_dir():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        print(f"agent0 import root: {p}")
        break
else:
    raise RuntimeError(
        "Could not locate agent0/ above this notebook. "
        "Clone the repo and place this notebook under src/agent0/notebooks/, "
        "or adjust sys.path manually."
    )

## Imports

In [ ]:
from dataclasses import asdict

from agent0.fast_app.ingest import normalize_record
from agent0.fast_app.type_detect import classify_record
from agent0.fast_app.fast_translate import attempt_fast_path
from agent0.fast_app.quality_gate import gate_translation
from agent0.shared.schemas import InputRecord, SequenceType

## Sample input

Five records exercising every branch of the fast path. AA outputs are sized
above `LENGTH_MIN_AA = 50`.

In [ ]:
records = [
    InputRecord(
        record_id="clean_dna",
        sequence="ATG" + "GCT" * 60 + "TAA",   # M + 60 A + *  (clean CDS)
    ),
    InputRecord(
        record_id="clean_rna",
        sequence="AUG" + "GCU" * 60 + "UAA",   # same protein, RNA alphabet
    ),
    InputRecord(
        record_id="protein_passthrough",
        sequence="MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQ" * 3,
    ),
    InputRecord(
        record_id="missing_start_codon",
        sequence="GCT" + "GCT" * 60 + "TAA",   # no ATG → fast path fails, slow needed
    ),
    InputRecord(
        record_id="ambiguous_garbage",
        sequence="X" * 200,                     # neither AA-canonical nor IUPAC-nuc
    ),
]

for r in records:
    print(f"{r.record_id:<22}  len={len(r.sequence):<4}  head={r.sequence[:30]}")

## Stage 1 — normalize

Uppercase, strip whitespace, strip alignment gaps. Audit trail is recorded.

In [ ]:
normalized = [normalize_record(r) for r in records]
for n in normalized:
    print(f"{n.record_id:<22}  transformations={n.transformations}")

## Stage 2 — type detect

Pure alphabet-composition analysis. DNA / RNA / protein / ambiguous.

In [ ]:
typed = [classify_record(n) for n in normalized]
for t in typed:
    print(f"{t.record_id:<22}  type={t.sequence_type.value}")

## Stage 3 — fast-path translation

Eligibility: DNA/RNA, length divisible by 3, ATG start, terminal stop, no internal
stops, all canonical bases. Failures are routed to the slow path.

In [ ]:
for t in typed:
    success, aa, dna_used = attempt_fast_path(t)
    print(
        f"{t.record_id:<22}  fast_path_success={success}  "
        f"aa_len={len(aa)}  dna_used_len={len(dna_used)}"
    )

## Stage 4 — quality gate

Length bounds, ambiguity-residue checks (terminal, run, fraction). Demonstrated
on a synthetic AA string with a long ambiguity run.

In [ ]:
samples = {
    "clean":          "M" * 200,
    "too_short":      "M" * 30,
    "long_x_run":     "M" * 50 + "XXXXX" + "M" * 50,
    "x_at_terminus":  "X" + "M" * 100,
}
for name, aa in samples.items():
    passed, reason, detail = gate_translation(aa)
    reason_str = reason.value if reason else "-"
    print(f"{name:<16}  passed={passed}  reason={reason_str}  detail={detail}")

## Full fast path via `process_record.local()`

`process_record` is decorated with `@app.function`. Calling `.local(...)` runs the
function body in the current process — same code path the deployed container would
execute, no Modal token or deploy required.

In [ ]:
from agent0.fast_app.modal_app import process_record

results = [process_record.local(asdict(r)) for r in records]

for r, res in zip(records, results):
    kind = res["kind"]
    payload = res["payload"]
    if kind == "translated":
        print(
            f"{r.record_id:<22}  {payload['verdict']:<20}  "
            f"aa_len={len(payload['aa_sequence'])}"
        )
    elif kind == "rejected":
        print(f"{r.record_id:<22}  REJECTED              reason={payload['reason']}")
    elif kind == "slow_needed":
        print(
            f"{r.record_id:<22}  SLOW_PATH_NEEDED      "
            f"dna_len={len(payload['dna_sequence'])}"
        )

## Slow path

Records emerging from the fast path with `kind == "slow_needed"` carry a post-RNA→DNA
sequence ready for ORF enumeration + ESM-2 perplexity scoring. That stage needs a GPU
and isn't covered here — the modules live under `agent0/slow_app/`.